# Step 2 — XGBoost on hand-crafted spike metrics (AD vs WT)

| | |
|---|---|
| **Original** | `Liam_work/tests/non_behavioral_spike_analysis/xgboost_spike_classification.ipynb` |
| **Status** | **Main analysis #1** (last edited 2026-09-25). |

**Question.** From a single neuron's spike-timing statistics, can we tell whether it came from an
APP/PS1 ("AD") or WT mouse — and does that work better in some sleep states, brain regions or cell types?

**Input.** Step-1 tables: 10 metrics × 3 states per neuron (`min_isis` and `mean_ifr` dropped; see cell
"Load"). Feature sets: `wake_*`, `nrem_*`, `rem_*` (10 each) or `combined` (30).

**Label.** `genotype`: APP/PS1 → AD (1), WT → 0. Age is **not** part of the label; it defines cohorts:
`ALL`, `OLD` (> 200 d) and `YOUNG` (≤ 200 d), each classified AD vs WT separately.

**Experiment grid** (each × ALL / OLD / YOUNG):

| family | configs | features used |
|---|---|---|
| Sleep state | All States, Wake, NREM, REM | only that state's 10 metrics (30 for All States) |
| Brain region | ACad, CA1, RSPv, V1 | neurons from that region, all 30 features |
| Cell type | RSU, FS | neurons of that type, all 30 features |

**Validation — leave-two-animals-out (L2O).** Each fold holds out one WT + one AD animal; every
(WT, AD) pair is a fold (9 × 9 = 81 folds for ALL). An animal therefore appears in several folds;
its per-animal score is the mean P(AD) over those folds. Models are class-balanced via sample weights.

**Hyperparameters.** One hyperopt TPE search (80 evals, 3-fold StratifiedGroupKFold by animal) on all
analysed neurons, combined features. These animals are also the L2O test animals, so report results as
"hyperparameters selected using all animals". The same params are reused for every config.

**Exclusions.** `caf104` dropped everywhere (81 % of neurons have within-epoch ISIs > 100 s — sorting
instability). `caf42` (WT, 207 d — right at the age cut) is additionally dropped from tuning and every
L2O run ("at user request; rationale pending").

**Metrics reported.** `N_Acc` / `N_BAcc` = neuron-level (balanced) accuracy pooled over folds;
`A_Acc` = fraction of unique animals whose mean P(AD) falls on the correct side of 0.5.

**Figures written (next to this notebook):** `fig_xgb_hyperopt.png`, `fig_xgb_perfold_boxplot.png`,
`fig_xgb_feature_importance.png`, `fig_xgb_spearman_states.png`, `fig_xgb_animal_{state,region,celltype}_*.png`,
`fig_xgb_age_confidence.png`.

**Runtime.** ~4 min tuning + a few minutes of L2O (20 parallel folds).

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

# Spike-metric classification: APP/PS1 vs WT
Three cohorts: all animals, older (>200 days), younger (<=200 days).
Each cohort compares sleep-state inputs, brain regions, and cell types.
Validation holds out one WT and one AD animal per fold (L2O); hyperparameters come
from one hyperopt search on all analysed animals. This notebook requires the
corrected `clean_spikemetrics_*.pkl` tables with explicit LvR R=5 ms.


Validation holds out one WT and one AD animal per fold (L2O). Hyperparameters come
from one hyperopt search on all analysed animals, which include the L2O test
animals: report as "hyperparameters selected using all animals". Scores are uncalibrated model
outputs. Age trends and importance-rank correlations are descriptive.
`caf42` remains excluded at user request; its biological rationale is pending.
Sex coloring does not adjust for sex, and these data do not establish progression.
Neuron rows can span multiple recording ages; animal IDs define every split.
The two representations retain their different ISI support (MLP: 1 ms–60 s;
metric tables: all positive finite ISIs), so this is not a model-only comparison.


## 1. Setup
Lab preamble, imports, the shared figure style (palettes + small plotting helpers), and the global knobs: `AGE_CUTOFF_DAYS = 200`, `N_JOBS = 20`, `EXCLUDE_ANIMALS = ['caf104']` (dropped everywhere), `SUBSET_EXCLUDE = ['caf42']` (dropped from tuning and every L2O run).

In [ ]:
# =============================================================================
# LAB-WIDE PREAMBLE — CPU thread limits + figure-export font config
# (Set BEFORE importing numpy/scipy/sklearn/xgboost so BLAS picks them up)
# =============================================================================
import os
os.environ['OMP_NUM_THREADS']        = '4'
os.environ['OPENBLAS_NUM_THREADS']   = '4'
os.environ['MKL_NUM_THREADS']        = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS']    = '4'
os.environ['PYTHONWARNINGS']         = 'ignore'   # also suppress in joblib workers

import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family': 'DejaVu Sans'})

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import pickle
import os
import warnings
import os

warnings.filterwarnings('ignore')

# =============================================================================
# FIGURE STYLE — matches the lab's prey-capture paper (Schneider, McGregor et al.):
# Arial, open axes, no gridlines, light-grey boxes with colored per-animal points,
# black-edged legends and statistic boxes, panel letters, captions printed below.
# =============================================================================
import matplotlib.lines as mlines
from matplotlib.patches import Patch

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': False,
    'axes.linewidth': 0.8, 'axes.edgecolor': 'black', 'axes.titlepad': 6,
    'xtick.major.size': 3, 'ytick.major.size': 3, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.minor.size': 1.5, 'ytick.minor.size': 1.5, 'xtick.minor.width': 0.6, 'ytick.minor.width': 0.6,
    'lines.linewidth': 1.5, 'patch.linewidth': 0.8,
    'legend.frameon': True, 'legend.fancybox': True, 'legend.edgecolor': 'black',
    'legend.framealpha': 1.0, 'legend.handlelength': 1.2, 'legend.handletextpad': 0.5,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight', 'savefig.facecolor': 'white',
})

# Colour-blind-checked palettes (CVD separation >= 12 ΔE for every pair)
GENO_COLORS  = {'WT': '#5B84B1', 'AD': '#D9826B'}                  # steel blue / terracotta
SEX_COLORS   = {('WT', 'M'): '#264F7D', ('WT', 'F'): '#8DB6E0',
                ('AD', 'M'): '#9E4631', ('AD', 'F'): '#F2A88C'}
STATE_COLORS = {'wake': '#E8B33A', 'nrem': '#7E5BA6', 'rem': '#2F8F5B'}
STATE_NAMES  = {'wake': 'Wake', 'nrem': 'NREM', 'rem': 'REM'}
GROUP_COLORS = {'Sleep State': '#3C3C3C', 'Brain Region': '#2F9BB5', 'Cell Type': '#BDBDBD'}
BOX_FILL, BOX_EDGE, REF_GREY = '#EBEBEB', '#6E6E6E', '#9E9E9E'


def panel_letters(fig, axes, letters='ABCDEFGHIJKL'):
    """Paper-style panel letters at the top-left of each panel (call once layout is final)."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_fig = fig.transFigure.inverted()
    for letter, ax in zip(letters, np.ravel(axes)):
        box = to_fig.transform_bbox(ax.get_tightbbox(renderer))
        fig.text(box.x0, box.y1 + 0.01, letter, fontsize=12, ha='left', va='bottom')


def stat_box(ax, text, x=0.97, y=0.03, ha='right', va='bottom'):
    """Square black-edged annotation box, as used for statistics in the paper."""
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va=va, fontsize=7.5, linespacing=1.3,
            bbox=dict(boxstyle='square,pad=0.4', facecolor='white', edgecolor='black', linewidth=0.8))


def genotype_handles(marker='o'):
    """Legend entries for the two genotypes."""
    return [mlines.Line2D([], [], color=GENO_COLORS[g], marker=marker, ls='', ms=5, label=lab)
            for g, lab in [('AD', 'AD (APP/PS1)'), ('WT', 'WT')]]


CHANCE_HANDLE = mlines.Line2D([], [], color='black', ls='--', lw=0.8, label='Chance (50%)')


AGE_CUTOFF_DAYS = 200  # young vs old boundary

print('Imports loaded.')
N_JOBS = 20  # parallel L2O folds (use 20 of 104 cores)
SUBSET_EXCLUDE = ['caf42']  # retained at user request; biological rationale pending
EXCLUDE_ANIMALS = ['caf104']  # documented long-ISI QC concern


## 2. Load the clean spike metrics and build the feature table
Loads the three per-state tables from step 1, checks their provenance tag and that neuron rows line up across states, drops `caf104` and the `min_isis` / `mean_ifr` features, then builds one wide table: one row per neuron with `nrem_*`, `wake_*`, `rem_*` columns. `STATE_FEATURES` maps each feature-set name to its columns. Labels: `genotype` (AD/WT), `age_group` (old/young), `group`.

In [ ]:
# =============================================================================
# LOAD PRE-COMPUTED CLEAN SPIKE METRICS
# =============================================================================
# These files are created by compute_clean_spikemetrics.ipynb (run it first).
# They contain metrics recomputed from raw ISIs with negative values filtered out.

# Repo copy: read the clean metric tables from the project's data/ folder.
DATA_DIR = '/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/data/spike_metrics/'

META_COLS = ['animal', 'epoch', 'cell_no', 'age', 'sex', 'geno', 'brain_region', 'cell_type']

# ---- EXCLUSION CONFIG (edit these to add/remove) ----

# Animals to exclude:
#   caf104 — spike sorting instability. 81% of its neurons have within-epoch
#   ISIs > 100s (next closest animal: 9.7%). This inflates max_isis, sd, cv,
#   and dist_alpha, making caf104 a massive outlier that dominates the classifier.
# EXCLUDE_ANIMALS is configured in setup.

# Features to drop:
#   min_isis  — measures recording sampling rate, not biology. ~70% of neurons
#               sit at exactly the hardware floor (0.001s or 0.00056s).
#   mean_ifr  — reciprocal of mean_isis, adds nothing.
DROP_FEATURES = ['min_isis', 'mean_ifr']

# All metrics from the pkl files (before dropping)
ALL_METRICS = ['mean_isis', 'mean_ifr', 'median_isis', 'max_isis', 'min_isis',
               'sd', 'cv', 'cv2', 'lv', 'lvr', 'dist_alpha', 'dist_beta']

# Metrics that actually get used (after dropping)
METRIC_NAMES = [m for m in ALL_METRICS if m not in DROP_FEATURES]

# ---- LOAD ----

state_dfs = {}
for state in ['nrem', 'wake', 'rem']:
    path = os.path.join(DATA_DIR, f'clean_spikemetrics_{state}.pkl')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{path} not found. Run compute_clean_spikemetrics.ipynb first!')
    state_dfs[state] = pd.read_pickle(path)
    if state_dfs[state].attrs.get('metric_schema') != 'isi-v2-lvr5ms':
        raise ValueError(f'{path}: corrected metric provenance missing; rebuild with the corrected notebook.')
    missing_ids = set(META_COLS) - set(state_dfs[state].columns)
    if missing_ids:
        raise ValueError(f'{path}: missing metadata/identity columns: {missing_ids}')
    print(f'Loaded {state}: {state_dfs[state].shape}')

# ---- EXCLUDE ANIMALS ----

if EXCLUDE_ANIMALS:
    for state in state_dfs:
        before = len(state_dfs[state])
        state_dfs[state] = state_dfs[state][~state_dfs[state]['animal'].isin(EXCLUDE_ANIMALS)]
        state_dfs[state] = state_dfs[state].reset_index(drop=True)
    after = len(state_dfs['nrem'])
    print(f'\nExcluded animals {EXCLUDE_ANIMALS}: {before} -> {after} neurons')

if DROP_FEATURES:
    print(f'Dropped features: {DROP_FEATURES}')
    print(f'Remaining features per state: {len(METRIC_NAMES)} ({METRIC_NAMES})')

# ---- VERIFY ALIGNMENT ----

for state in ['wake', 'rem']:
    if not state_dfs['nrem'][META_COLS].equals(state_dfs[state][META_COLS]):
        raise ValueError(f'Neuron identities or metadata do not align between NREM and {state}.')
    finite = state_dfs[state][['lv', 'lvr']].dropna()
    if len(finite) and np.allclose(finite['lv'], finite['lvr'], rtol=0, atol=1e-12):
        raise ValueError(f'{state}: LvR still duplicates LV; verify the corrected metric build.')

# ---- BUILD UNIFIED DATAFRAME ----

df = state_dfs['nrem'][META_COLS].copy()

STATE_FEATURES = {}
for state in ['nrem', 'wake', 'rem']:
    state_cols = []
    for col in METRIC_NAMES:
        new_col = f'{state}_{col}'
        df[new_col] = state_dfs[state][col].values
        state_cols.append(new_col)
    STATE_FEATURES[state] = state_cols

COMBINED_COLS = []
for state in ['nrem', 'wake', 'rem']:
    COMBINED_COLS.extend(STATE_FEATURES[state])
STATE_FEATURES['combined'] = COMBINED_COLS

print(f'\nFeature sets:')
for state, cols in STATE_FEATURES.items():
    print(f'  {state:10s}: {len(cols)} features')

# ---- LABELS ----

# CLASSIFICATION TARGET = genotype only (APP/PS1 -> 'AD', WT -> 'WT').
# Age is NOT in the label; 'age_group' is only used to split cohorts (ALL / OLD / YOUNG).
df['genotype'] = df['geno'].map(lambda x: 'AD' if 'app' in str(x).lower() else 'WT')
df['age_group'] = df['age'].apply(lambda x: 'old' if x > AGE_CUTOFF_DAYS else 'young')
df['group'] = df['age_group'] + '_' + df['genotype']

# ---- CLEAN NaN/inf ----

n_before = len(df)
df = df.dropna(subset=COMBINED_COLS)
print(f'\nDropped {n_before - len(df)} rows with NaN metrics ({len(df)} remaining)')

mask = np.isfinite(df[COMBINED_COLS].values).all(axis=1)
df = df[mask].copy()
print(f'After removing inf: {len(df)} neurons')

# ---- SUMMARY ----

print(f'\n{"="*60}')
print(f'DATASET SUMMARY')
print(f'{"="*60}')
print(f'Total neurons: {len(df)}')
print(f'Excluded animals: {EXCLUDE_ANIMALS}')
print(f'Dropped features: {DROP_FEATURES}')
print(f'\nBy group:')
for group in sorted(df['group'].unique()):
    sub = df[df['group'] == group]
    animals = sorted(sub['animal'].unique())
    print(f'  {group:12s}: {len(sub):4d} neurons, {len(animals)} animals: {animals}')

print(f'\nAge cutoff: {AGE_CUTOFF_DAYS} days')
print(f'\nAnimal ages:')
for animal in sorted(df['animal'].unique()):
    sub = df[df['animal'] == animal]
    print(f'  {animal:8s}: age={sub["age"].iloc[0]:7.1f}d  geno={sub["genotype"].iloc[0]:2s}  '
          f'group={sub["group"].iloc[0]:10s}  n={len(sub)}')

## 3. Hyperparameter search (hyperopt / TPE)
Search space adapted from the lab reference notebook (`classify_RF_AS-latest.ipynb`, not in this repo) but narrowed to sane ranges. Objective = mean balanced accuracy over 3 grouped folds (no animal in both train and validation). Also scores the old untuned defaults on the same folds for comparison. Result: `BEST_XGB_PARAMS`, passed explicitly into every L2O worker.

In [ ]:
# =============================================================================
# XGBOOST — Hyperopt Hyperparameter Tuning  (lab convention)
# =============================================================================
# Uses hyperopt (TPE search) — the lab's standard tuning library.
# Search space matches the one used in tests/classify_RF_AS-latest.ipynb.
# Inner CV: 3-fold StratifiedGroupKFold (stratify by genotype, group by animal),
# with the 3 folds of each trial trained in parallel. Tuned on all analysed
# animals (caf42 excluded), so the L2O test animals below are part of this
# search: report as "hyperparameters selected using all animals".
# BEST_XGB_PARAMS is passed explicitly to every L2O fold.

import time
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import balanced_accuracy_score
from joblib import Parallel, delayed

# --- knobs -------------------------------------------------------------------
MAX_EVALS          = 80       # James ran 1000 in the reference notebook — 80 is plenty for narrowing
HYPEROPT_CV_SPLITS = 3
RANDOM_SEED        = 42


def fit_xgb(params, X_train, y_train, seed=42):
    """Class-balanced XGBoost (WT=0, AD=1), one thread.

    `params` is passed in explicitly because parallel workers cannot see notebook globals.
    """
    model = xgb.XGBClassifier(**params, objective='binary:logistic', eval_metric='logloss',
                              random_state=seed, verbosity=0, n_jobs=1)
    model.fit(X_train, y_train, sample_weight=compute_sample_weight('balanced', y_train))
    return model


def _cast_int_params(p):
    """hyperopt returns floats for quniform — cast back to int where XGBoost expects."""
    out = dict(p)
    for k in ('n_estimators', 'max_depth', 'min_child_weight'):
        if k in out:
            out[k] = int(out[k])
    return out


# --- prep data ---------------------------------------------------------------
df_opt = df[~df['animal'].isin(SUBSET_EXCLUDE)].reset_index(drop=True)
feat_cols_opt = STATE_FEATURES['combined']
X_opt = df_opt[feat_cols_opt].values.astype(float)
y_opt = (df_opt['genotype'] == 'AD').astype(int).values
groups_opt = df_opt['animal'].values
cv_splits = list(StratifiedGroupKFold(n_splits=HYPEROPT_CV_SPLITS, shuffle=True,
                                      random_state=RANDOM_SEED).split(X_opt, y_opt, groups_opt))

print(f'Hyperopt setup: {len(df_opt)} neurons, {df_opt["animal"].nunique()} animals, '
      f'{len(feat_cols_opt)} features, {MAX_EVALS} evals.')
print(f'Inner CV: {HYPEROPT_CV_SPLITS}-fold StratifiedGroupKFold.')

# --- search space (matches lab reference notebook) ---------------------------
space = {
    'learning_rate':    hp.uniform('learning_rate', 0.01, 0.3),     # narrower & sane (James's 1e-1..1e4 was problematic)
    'n_estimators':     hp.quniform('n_estimators', 100, 350, 5),
    'max_depth':        hp.quniform('max_depth', 3, 8, 1),
    'gamma':            hp.quniform('gamma', 0, 0.1, 0.01),
    'reg_alpha':        hp.uniform('reg_alpha', 0, 5),         # sane range — James's 0-20 was OK
    'reg_lambda':       hp.uniform('reg_lambda', 0, 10),        # sane range — James's 0-1e8 was destroying the model
    'subsample':        hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
}


def _cv_fold_score(params, X_tr, y_tr, X_va, y_va):
    """Balanced accuracy of one inner-CV fold."""
    return balanced_accuracy_score(y_va, fit_xgb(params, X_tr, y_tr).predict(X_va))


def xgb_objective(params):
    p = _cast_int_params(params)
    scores = Parallel(n_jobs=HYPEROPT_CV_SPLITS)(
        delayed(_cv_fold_score)(p, X_opt[tr], y_opt[tr], X_opt[va], y_opt[va])
        for tr, va in cv_splits)
    # hyperopt minimizes — return negative score
    return {'loss': -float(np.mean(scores)), 'status': STATUS_OK}


# Untuned reference: the defaults this notebook used before tuning, on the same CV folds.
DEFAULT_XGB_PARAMS = dict(n_estimators=200, max_depth=4, learning_rate=0.1,
                          subsample=0.8, colsample_bytree=0.8, min_child_weight=5)
default_score = float(np.mean(Parallel(n_jobs=HYPEROPT_CV_SPLITS)(
    delayed(_cv_fold_score)(DEFAULT_XGB_PARAMS, X_opt[tr], y_opt[tr], X_opt[va], y_opt[va])
    for tr, va in cv_splits)))
print(f'Default settings (200 trees, depth 4, lr 0.1): balanced_acc = {default_score:.4f}')

# --- run search --------------------------------------------------------------
print(f'Running {MAX_EVALS} hyperopt trials...')
t0 = time.time()
trials = Trials()
best_raw = fmin(
    fn        = xgb_objective,
    space     = space,
    algo      = tpe.suggest,
    max_evals = MAX_EVALS,
    trials    = trials,
    rstate    = np.random.default_rng(RANDOM_SEED),
)
BEST_XGB_PARAMS = _cast_int_params(space_eval(space, best_raw))
best_score = -min(t['result']['loss'] for t in trials.trials)

print(f'\nSearch finished in {time.time() - t0:.0f}s')
print(f'Best trial: balanced_acc = {best_score:.4f}')
print('Best hyperparameters:')
for k, v in BEST_XGB_PARAMS.items():
    print(f'  {k:<20} = {v:.5g}' if isinstance(v, float) else f'  {k:<20} = {v}')

# --- visualize convergence ---------------------------------------------------
neg_scores = [-t['result']['loss'] for t in trials.trials]
best_so_far = np.maximum.accumulate(neg_scores)

trial_no = np.arange(1, len(neg_scores) + 1)
fig, ax = plt.subplots(figsize=(3.4, 2.4))
ax.scatter(trial_no, neg_scores, s=12, color=REF_GREY, linewidths=0, label='Trial')
ax.plot(trial_no, best_so_far, color='black', lw=1.5, label='Running best')
ax.axhline(default_score, color=REF_GREY, ls='--', lw=1.0, label='Default settings')
ax.set_xlabel('Hyperopt trial')
ax.set_ylabel(f'Balanced accuracy\n({HYPEROPT_CV_SPLITS}-fold grouped CV)')
ax.set_title(f'XGBoost hyperparameter search\n'
             f'best {best_score:.3f} vs default {default_score:.3f} '
             f'({best_score - default_score:+.3f})', fontsize=8)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.savefig('fig_xgb_hyperopt.png')
plt.show()
print(f'Tuning gained {best_score - default_score:+.4f} balanced accuracy over the defaults.')
print('Saved: fig_xgb_hyperopt.png')


## 4. Leave-two-animals-out runs for every (age split × config)
`_split_df` slices neurons by age group and region / cell type; `run_xgb_l2o` builds every (WT, AD) held-out pair and trains the folds in parallel; each fold returns one record per held-out animal (its neurons' accuracy and mean P(AD)) plus gain importances. `_aggregate_by_animal_xgb` collapses the records to one row per unique animal. Ends with the per-animal accuracy box plot and the summary table (`N_Acc`, `N_BAcc`, `A_Acc`).

In [ ]:
# =============================================================================
# XGBOOST: Per-animal neuron accuracy by configuration  (LEAVE-TWO-ANIMAL-OUT)
# =============================================================================
# Each fold holds out 1 WT + 1 AD animal. For each config (sleep state /
# brain region / cell type), we train L2O across all WT-AD pairings.
# Boxplot shows ONE dot per unique animal (mean accuracy across its pairs).
# caf42 is excluded from this comprehensive subset analysis.

import matplotlib.lines as mlines
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from joblib import Parallel, delayed

if 'df' not in globals() or 'STATE_FEATURES' not in globals():
    raise RuntimeError('Run data loading + framework cells first.')

# SUBSET_EXCLUDE is configured in setup.
print(f'Animals excluded from subset analysis: {SUBSET_EXCLUDE}')

AGE_SPLITS = ['ALL', 'OLD', 'YOUNG']
AGE_LABELS = {'ALL': 'All Animals', 'OLD': 'Old (>200d)', 'YOUNG': 'Young (≤200d)'}

# (family, display_name, category, key)
CONFIGS = [
    ('Sleep State', 'REM',        'state',     'rem'),
    ('Sleep State', 'All States', 'state',     'combined'),
    ('Sleep State', 'Wake',       'state',     'wake'),
    ('Sleep State', 'NREM',       'state',     'nrem'),
    ('Brain Region', 'ACad',      'region',    'ACad'),
    ('Brain Region', 'CA1',       'region',    'CA1'),
    ('Brain Region', 'RSPv',      'region',    'RSPv'),
    ('Brain Region', 'V1',        'region',    'V1'),
    ('Cell Type',    'FS',        'cell_type', 'FS'),
    ('Cell Type',    'RSU',       'cell_type', 'RSU'),
]

# GENO_COLORS, GROUP_COLORS and the plotting helpers come from the figure-style block in setup.

def _train_one_l2o_fold_xgb(train_idx, test_idx, test_animals, X, y, groups, params):
    """One L2O fold for XGBoost (WT=0, AD=1). Returns per-held-out-animal records
    and the fold model's gain importance."""
    model = fit_xgb(params, X[train_idx], y[train_idx])
    preds_all = model.predict(X[test_idx])
    probs_all = model.predict_proba(X[test_idx])[:, 1]

    test_groups = groups[test_idx]
    records = []
    for ta in test_animals:
        mask = test_groups == ta
        preds, probs, y_anim = preds_all[mask], probs_all[mask], y[test_idx][mask]
        records.append({
            'animal': ta,
            'geno':   'AD' if y_anim[0] == 1 else 'WT',
            'true':   int(y_anim[0]),
            'pred':   int(np.mean(probs) >= 0.5),
            'mean_prob': float(np.mean(probs)),
            'neuron_acc': float(accuracy_score(y_anim, preds)),
            'n_neurons':  int(len(y_anim)),
        })
    return records, np.asarray(model.feature_importances_, dtype=float)


def run_xgb_l2o(df_sub, feat_cols):
    """Run XGBoost L2O over all (WT, AD) animal pairs with BEST_XGB_PARAMS.

    Returns per-held-out-animal records (each animal appears once per
    opposite-genotype partner) and gain importance averaged across folds,
    or None if either genotype has fewer than 2 animals.
    """
    work = df_sub[~df_sub['animal'].isin(SUBSET_EXCLUDE)].reset_index(drop=True)
    counts = work.groupby('genotype')['animal'].nunique()
    if len(counts) != 2 or counts.min() < 2:
        return None

    X = work[feat_cols].values.astype(float)
    y = (work['genotype'] == 'AD').astype(int).values
    groups = work['animal'].values

    # Build L2O pairs (one WT + one AD per fold)
    folds = []
    for wt in sorted(work.loc[y == 0, 'animal'].unique()):
        for ad in sorted(work.loc[y == 1, 'animal'].unique()):
            test_mask = (groups == wt) | (groups == ad)
            folds.append((np.where(~test_mask)[0], np.where(test_mask)[0], (wt, ad)))

    fold_outputs = Parallel(n_jobs=N_JOBS)(
        delayed(_train_one_l2o_fold_xgb)(tr, te, tas, X, y, groups, BEST_XGB_PARAMS)
        for tr, te, tas in folds
    )
    importances = np.vstack([imp for _recs, imp in fold_outputs])
    return {'records': [r for recs, _imp in fold_outputs for r in recs],
            'mean_importance': importances.mean(axis=0),
            'std_importance': importances.std(axis=0),
            'feature_cols': list(feat_cols)}


def _aggregate_by_animal_xgb(fold_res):
    """Collapse per-animal-eval records to one record per unique animal.

    `mean_prob` is P(AD). The 'true' / 'pred' fields use LabelEncoder
    integer values (WT=0, AD=1). We derive
    those integer values from the fold records so the comparison is correct
    regardless of encoding direction.
    """
    if not fold_res:
        return []
    # Find the integer value that corresponds to AD (and WT) in this run
    ad_int = next((r['true'] for r in fold_res if r['geno'] == 'AD'), 1)
    wt_int = next((r['true'] for r in fold_res if r['geno'] == 'WT'), 1 - ad_int)

    by_animal = {}
    for f in fold_res:
        by_animal.setdefault(f['animal'], []).append(f)
    out = []
    for animal, recs in by_animal.items():
        probs = np.array([r['mean_prob'] for r in recs])
        accs  = np.array([r['neuron_acc'] for r in recs])
        mean_prob = float(np.mean(probs))   # P(AD)
        animal_pred = ad_int if mean_prob >= 0.5 else wt_int
        out.append({
            'animal': animal,
            'geno': recs[0]['geno'],
            'true': recs[0]['true'],
            'pred': animal_pred,
            'mean_prob': mean_prob,
            'neuron_acc': float(np.mean(accs)),
            'n_neurons': int(recs[0]['n_neurons']),
            'n_folds': len(recs),
        })
    return out

def _get_metrics_xgb(fold_res):
    """Neuron-level + per-unique-animal metrics from a flat list of L2O records.

    n_acc and n_bacc are derived from each record's `neuron_acc` and `n_neurons`.
    Since every fold record corresponds to ONE held-out animal (all of whose
    neurons share a class), the animal's neuron_acc equals the model's recall
    for that animal's class, weighted by its n_neurons.
    """
    if not fold_res:
        return None

    # Per-class (recall) computation by weighted sum
    by_class = {0: {'correct': 0.0, 'total': 0}, 1: {'correct': 0.0, 'total': 0}}
    total_correct = 0.0
    total_n = 0
    for r in fold_res:
        n = r['n_neurons']
        c = int(r['true'])
        by_class.setdefault(c, {'correct': 0.0, 'total': 0})
        by_class[c]['correct'] += r['neuron_acc'] * n
        by_class[c]['total']   += n
        total_correct          += r['neuron_acc'] * n
        total_n                += n

    n_acc = total_correct / total_n if total_n else float('nan')

    recalls = []
    for c in by_class:
        if by_class[c]['total'] > 0:
            recalls.append(by_class[c]['correct'] / by_class[c]['total'])
    n_bacc = float(np.mean(recalls)) if recalls else float('nan')

    agg = _aggregate_by_animal_xgb(fold_res)
    a_correct = sum(1 for r in agg if r['true'] == r['pred'])
    a_total = len(agg)

    return {
        'n_acc': n_acc, 'n_bacc': n_bacc,
        'a_acc': a_correct / a_total if a_total else float('nan'),
        'a_correct': a_correct, 'a_total': a_total,
    }



def _split_df(df_full, age_split, category, key):
    """Slice df by age + region/cell-type, return None if not enough animals."""
    d = df_full.copy()
    if age_split == 'OLD':
        d = d[d['age_group'] == 'old']
    elif age_split == 'YOUNG':
        d = d[d['age_group'] == 'young']
    if category == 'region':
        d = d[d['brain_region'] == key]
    elif category == 'cell_type':
        d = d[d['cell_type'] == key]
    d = d[~d['animal'].isin(SUBSET_EXCLUDE)]
    if d['genotype'].nunique() < 2:
        return None
    if d.groupby('genotype')['animal'].nunique().min() < 2:
        return None
    return d


# ── Train L2O for every (age_split × config) combination ────────────────────
print('\nTraining XGBoost L2O across all configurations.')
print('Each animal appears in multiple folds (one per opposite-genotype partner).')

xgb_runs = {}
l2o_results = {}              # (age_split, category, key) -> flat list of per-animal records
l2o_feature_importances = {}  # (age_split, category, key) -> {features, importance}
for age_split in AGE_SPLITS:
    print(f'\n══ {age_split} ══')
    for family, display_name, category, key in CONFIGS:
        d = _split_df(df, age_split, category, key)
        if d is None:
            print(f'  {display_name:<12} skipped (needs >=2 animals per genotype)')
            continue
        if category == 'state':
            feat_cols = [c for c in STATE_FEATURES[key] if c in d.columns]
        else:
            feat_cols = [c for c in STATE_FEATURES['combined'] if c in d.columns]
        out = run_xgb_l2o(d, feat_cols)
        if out:
            xgb_runs[(age_split, category, key)] = out
            records = out['records']
            l2o_results[(age_split, category, key)] = records
            l2o_feature_importances[(age_split, category, key)] = {
                'features':   out['feature_cols'],
                'importance': out['mean_importance'],
                'std': out['std_importance'],
            }
            agg = _aggregate_by_animal_xgb(records)
            n_correct = sum(1 for r in agg if r['true'] == r['pred'])
            print(f'  {display_name:<12} A_Acc {n_correct}/{len(agg)} '
                  f'({100*n_correct/len(agg):.1f}%) over {len(records)} animal-evals')
        else:
            l2o_results[(age_split, category, key)] = None

# ── Boxplot: one dot per UNIQUE animal (mean neuron acc across pairs) ──────
print('\nPer-animal neuron accuracy by configuration. Each dot = one held-out animal (mean across its '
      'L2O pairs); box = IQR, whiskers = min/max, number = median, dashed line = chance.')
_rng = np.random.RandomState(7)
fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), sharey=True)

for ax, age_split in zip(axes, AGE_SPLITS):
    rows = []
    for family, display_name, category, key in CONFIGS:
        fold_res = l2o_results.get((age_split, category, key))
        if not fold_res:
            continue
        agg = _aggregate_by_animal_xgb(fold_res)
        accs = np.array([r['neuron_acc'] for r in agg])
        rows.append({
            'family': family,
            'name': display_name,
            'median': float(np.median(accs)),
            'animal_accs': accs,
            'animal_res':  agg,
        })
    rows.sort(key=lambda r: r['median'], reverse=True)

    for i, row in enumerate(rows):
        ax.boxplot(
            row['animal_accs'], positions=[i], widths=0.6,
            patch_artist=True, showfliers=False, whis=(0, 100),
            medianprops=dict(color='black', linewidth=1.2),
            boxprops=dict(facecolor=BOX_FILL, edgecolor=BOX_EDGE, linewidth=0.8),
            whiskerprops=dict(color=BOX_EDGE, linewidth=0.8),
            capprops=dict(color=BOX_EDGE, linewidth=0.8),
        )
        jitter = _rng.uniform(-0.18, 0.18, len(row['animal_res']))
        ax.scatter(i + jitter, [r['neuron_acc'] for r in row['animal_res']],
                   c=[GENO_COLORS[r['geno']] for r in row['animal_res']], s=10, linewidths=0, zorder=3)
        y_text = min(max(row['animal_accs']) + 0.03, 1.01)
        ax.text(i, y_text, f"{row['median']:.2f}", ha='center', va='bottom', fontsize=6.5)

    ax.axhline(0.5, color='black', ls='--', lw=0.8, zorder=0)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels([r['name'] for r in rows], rotation=45, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.6, len(rows) - 0.4)
    ax.set_ylim(0, 1.06)
    ax.set_title(AGE_LABELS[age_split])
axes[0].set_ylabel('Per-animal neuron accuracy')

fig.tight_layout(w_pad=1.0)
axes[-1].legend(handles=genotype_handles() + [CHANCE_HANDLE],
                loc='upper left', bbox_to_anchor=(1.03, 1.0))
panel_letters(fig, axes)
plt.savefig('fig_xgb_perfold_boxplot.png')
plt.show()
print('Saved: fig_xgb_perfold_boxplot.png')



# =============================================================================
# Summary table
# =============================================================================
print('\n' + '=' * 70)
print('SUMMARY TABLE')
print('=' * 70)
for age_split in AGE_SPLITS:
    print(f'\n  {AGE_LABELS[age_split]}')
    print(f'  {"Config":<15} {"N_Acc":>8} {"N_BAcc":>8} {"A_Acc":>8}  {"Animals":>8}')
    print(f'  {"-"*55}')
    for family, dname, category, key in CONFIGS:
        m = _get_metrics_xgb(l2o_results.get((age_split, category, key)))
        if not m:
            print(f'  {dname:<15} {"n/a":>8} {"n/a":>8} {"n/a":>8}')
            continue
        print(f'  {dname:<15} {m["n_acc"]:>7.1%} {m["n_bacc"]:>8.1%} {m["a_acc"]:>7.1%}  ({m["a_correct"]}/{m["a_total"]})')
    print()


## 5. Feature importance
Gain importance averaged over the L2O fold models. (A) top-10 features of the all-state model. (B) per-state models' importance profiles and Spearman ρ between their rankings. Gain = training split utility, not held-out importance or effect direction.

In [ ]:
# Descriptive training-gain importance from the same L2O fold models.
# Gain is model split utility, not held-out permutation importance or effect direction.
combined = xgb_runs[('ALL', 'state', 'combined')]
feat_cols = combined['feature_cols']
imp_l2o = np.asarray(combined['mean_importance'])
imp_l2o_std = np.asarray(combined['std_importance'])
imp_df = pd.DataFrame({
    'feature': feat_cols,
    'state':   [f.split('_')[0] for f in feat_cols],
    'metric':  ['_'.join(f.split('_')[1:]) for f in feat_cols],
    'mean':    imp_l2o,
    'std':     imp_l2o_std,
}).sort_values('mean', ascending=False).reset_index(drop=True)

STATE_COL = STATE_COLORS

# === Figure A: Combined-feature top 10, ranked, error bars across L2O folds ==
print('Top 10 features of the all-state XGBoost models: gain importance, mean ± SD across L2O folds '
      '(training split utility, not effect direction).')
top = imp_df.head(10).iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(3.8, 3.0))
ypos = np.arange(len(top))
ax.barh(ypos, top['mean'], xerr=top['std'], height=0.7,
        color=[STATE_COL[s] for s in top['state']], linewidth=0,
        error_kw=dict(elinewidth=0.8, ecolor='black', capsize=0))
ax.set_yticks(ypos)
ax.set_yticklabels([f'{STATE_NAMES[s]} {m}' for s, m in zip(top['state'], top['metric'])])
ax.set_xlim(left=0)
ax.set_xlabel('Gain importance (mean ± SD)')
ax.set_title('XGBoost, all-state model: top 10 features')
ax.legend(handles=[Patch(facecolor=c, label=STATE_NAMES[s]) for s, c in STATE_COL.items()],
          loc='lower right')
plt.savefig('fig_xgb_feature_importance.png')
plt.show()
print('Saved: fig_xgb_feature_importance.png')


# =============================================================================
# Per-state feature importance — small multiples + cross-state agreement
# =============================================================================
from scipy.stats import spearmanr

state_imp = {}
for state in ['wake', 'nrem', 'rem']:
    result = xgb_runs[('ALL', 'state', state)]
    state_imp[state] = {f.replace(f'{state}_', ''): imp
                        for f, imp in zip(result['feature_cols'], result['mean_importance'])}

common = sorted(set.intersection(*(set(d) for d in state_imp.values())))
imp_mat = pd.DataFrame({s: [state_imp[s][k] for k in common] for s in ['wake', 'nrem', 'rem']},
                       index=common)
imp_mat['_mean'] = imp_mat.mean(axis=1)
imp_mat = imp_mat.sort_values('_mean', ascending=False).drop(columns='_mean')

print('\nPer-state importance (sorted by mean):')
print(imp_mat.round(4))

pairs = [('wake', 'nrem'), ('wake', 'rem'), ('nrem', 'rem')]
rhos, pvals = [], []
for a, b in pairs:
    rho, pv = spearmanr(imp_mat[a], imp_mat[b])
    rhos.append(rho); pvals.append(pv)


def _draw_rank_agreement(ax, pairs, rhos, xlabel):
    """Lollipop of Spearman ρ for each pair of sleep states."""
    ypos = np.arange(len(pairs))
    ax.axvline(0, color='black', ls='--', lw=0.8)
    ax.hlines(ypos, 0, rhos, color=BOX_EDGE, lw=1.2)
    ax.scatter(rhos, ypos, s=24, color='black', zorder=3, linewidths=0)
    for y, rho in zip(ypos, rhos):
        ax.annotate(f'ρ = {rho:+.2f}', xy=(rho, y), xytext=(6 if rho >= 0 else -6, 0),
                    textcoords='offset points', ha='left' if rho >= 0 else 'right',
                    va='center', fontsize=7.5)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f'{STATE_NAMES[a]} vs {STATE_NAMES[b]}' for a, b in pairs])
    ax.set_ylim(len(pairs) - 0.4, -0.6)
    ax.set_xlim(-1, 1)
    ax.set_xticks([-1, -0.5, 0, 0.5, 1])
    ax.set_xlabel(xlabel)


print('A, gain importance of each metric in the per-state models (mean across L2O folds). '
      'B, Spearman ρ between the per-state rankings (descriptive; +1 = same ranking, −1 = opposite).')
fig = plt.figure(figsize=(7.2, 4.4), layout='constrained')
gs = fig.add_gridspec(2, 3, height_ratios=[2.2, 1])
state_axes = []
xmax = max(imp_mat.values.max() * 1.15, 0.001)
for c, state in enumerate(['wake', 'nrem', 'rem']):
    ax = fig.add_subplot(gs[0, c])
    s = imp_mat[state].sort_values(ascending=True)
    color = STATE_COL[state]
    ax.hlines(range(len(s)), 0, s.values, color=color, lw=1.2)
    ax.scatter(s.values, range(len(s)), color=color, s=18, zorder=3, linewidths=0)
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels(s.index)
    ax.set_xlim(0, xmax)
    ax.set_xlabel('Gain importance')
    ax.set_title(f'{STATE_NAMES[state]} model')
    state_axes.append(ax)
ax_s = fig.add_subplot(gs[1, :])
_draw_rank_agreement(ax_s, pairs, rhos, 'Spearman ρ between per-state feature-importance rankings')
panel_letters(fig, [state_axes[0], ax_s])
plt.savefig('fig_xgb_spearman_states.png')
plt.show()
print('Saved: fig_xgb_spearman_states.png')


## 6. Per-animal predictions, one figure per config
Top row: each held-out animal's mean P(AD) (sorted; outlined/× = misclassified) for ALL / OLD / YOUNG. Bottom row: that config's top-12 gain importances.

In [ ]:
# =============================================================================
# XGBOOST FIGURE: Per-animal predictions (AD vs WT)  (LEAVE-TWO-ANIMAL-OUT)
# One figure per condition — TOP row = per-animal P(AD), BOTTOM row = feature
# importance (averaged across the same L2O folds) for that age split.
# =============================================================================
# Reuses l2o_results + l2o_feature_importances from the L2O training cell.
# caf42 is excluded.

import matplotlib.lines as mlines

if 'l2o_results' not in globals() or 'l2o_feature_importances' not in globals():
    raise RuntimeError('Run the L2O training cell first — l2o_results + '
                       'l2o_feature_importances are required.')

ANIMAL_AGE = df[['animal', 'age']].drop_duplicates().groupby('animal')['age'].median().to_dict()
STATE_BAR_COLORS = STATE_COLORS
TOP_N_FEATURES = 12   # how many features to show in the importance row


def _plot_xgb_per_animal_condition(category, key, label, group_label, save_prefix):
    """2-row figure per condition: per-animal AD score on top, feature importance below."""
    print('\n' + '=' * 70)
    print(f'{group_label}: Per-Animal P(AD) + feature importance — {label}')
    print('=' * 70)
    print('Top: mean uncalibrated AD score per held-out animal (averaged across L2O pairs); outline and '
          '\u00d7 = misclassified. Bottom: gain importance averaged across the same L2O folds.')

    fig, axes = plt.subplots(2, len(AGE_SPLITS), figsize=(7.5, 6.4),
                             gridspec_kw={'height_ratios': [1.3, 1]})
    for c, age_split in enumerate(AGE_SPLITS):
        ax_top, ax_bot = axes[0, c], axes[1, c]
        fold_res = l2o_results.get((age_split, category, key))

        # ---------- TOP: per-animal AD score ----------
        ax_top.set_title(f'{label} \u00b7 {AGE_LABELS[age_split]}')
        ax_top.set_xlim(0, 1)
        if not fold_res:
            ax_top.text(0.5, 0.5, 'No data', ha='center', va='center',
                        transform=ax_top.transAxes, color=REF_GREY)
        else:
            agg = sorted(_aggregate_by_animal_xgb(fold_res), key=lambda f: f['mean_prob'])
            probs = [r['mean_prob'] for r in agg]
            wrong = [r['true'] != r['pred'] for r in agg]
            ypos = np.arange(len(agg))
            ax_top.barh(ypos, probs, height=0.7, color=[GENO_COLORS[r['geno']] for r in agg],
                        edgecolor=['black' if w else 'none' for w in wrong],
                        linewidth=[1.0 if w else 0 for w in wrong])
            for y, p, w in zip(ypos, probs, wrong):
                if w:
                    ax_top.text(min(p + 0.02, 0.95), y, '\u00d7', ha='left', va='center', fontsize=8)
            ax_top.set_yticks(ypos)
            ax_top.set_yticklabels([f"{r['animal']} ({int(ANIMAL_AGE.get(r['animal'], 0))} d)" for r in agg],
                                   fontsize=7)
            ax_top.axvline(0.5, color='black', ls='--', lw=0.8)
            ax_top.set_xlabel('Mean AD score (uncalibrated)')
            n_correct = sum(1 for r in agg if r['true'] == r['pred'])
            all_neuron_acc = _get_metrics_xgb(fold_res)['n_acc']
            ax_top.set_title(f'{label} · {AGE_LABELS[age_split]}\n'
                             f'{n_correct}/{len(agg)} animals, neuron acc. {all_neuron_acc:.1%}', fontsize=8)

        # ---------- BOTTOM: feature importance for this (age_split, condition) ----------
        imp = l2o_feature_importances.get((age_split, category, key))
        if not imp:
            ax_bot.text(0.5, 0.5, 'No importance', ha='center', va='center',
                        transform=ax_bot.transAxes, color=REF_GREY)
        else:
            order = np.argsort(imp['importance'])[::-1][:TOP_N_FEATURES]
            feats = [imp['features'][i] for i in order]
            vals  = [imp['importance'][i] for i in order]
            bar_colors = [STATE_BAR_COLORS.get(f.split('_')[0], '#888') for f in feats]
            ax_bot.barh(range(len(feats)), vals[::-1], color=bar_colors[::-1], height=0.7, linewidth=0)
            ax_bot.set_yticks(range(len(feats)))
            ax_bot.set_yticklabels(feats[::-1], fontsize=7)
            ax_bot.set_xlabel('Gain importance')
            ax_bot.set_title(f'Top {len(feats)} features')

    fig.tight_layout(w_pad=1.0, h_pad=1.5)
    axes[0, -1].legend(handles=[Patch(facecolor=GENO_COLORS['AD'], label='AD (APP/PS1)'),
                                Patch(facecolor=GENO_COLORS['WT'], label='WT'),
                                Patch(facecolor='white', edgecolor='black', label='Misclassified')],
                       loc='upper left', bbox_to_anchor=(1.03, 1.0))
    # state-color legend only meaningful for combined / region / cell-type (multi-state features)
    if category != 'state' or key == 'combined':
        axes[1, -1].legend(handles=[Patch(facecolor=c, label=f'{STATE_NAMES[s]} feature')
                                    for s, c in STATE_BAR_COLORS.items()],
                           loc='upper left', bbox_to_anchor=(1.03, 1.0))
    panel_letters(fig, axes[:, 0])
    plt.savefig(f'{save_prefix}.png')
    plt.show()
    print(f'Saved: {save_prefix}.png')


# Sleep states — one figure each
for state_name, key in [('All States', 'combined'), ('NREM', 'nrem'),
                        ('Wake', 'wake'), ('REM', 'rem')]:
    _plot_xgb_per_animal_condition(
        category='state', key=key, label=state_name,
        group_label='STATE', save_prefix=f'fig_xgb_animal_state_{key}'
    )

# Brain regions — one figure each
for region in ['ACad', 'CA1', 'RSPv', 'V1']:
    _plot_xgb_per_animal_condition(
        category='region', key=region, label=region,
        group_label='REGION', save_prefix=f'fig_xgb_animal_region_{region.lower()}'
    )

# Cell types — one figure each
for ct in ['FS', 'RSU']:
    _plot_xgb_per_animal_condition(
        category='cell_type', key=ct, label=ct,
        group_label='CELLTYPE', save_prefix=f'fig_xgb_animal_celltype_{ct.lower()}'
    )


## 7. Model score vs age (all animals, all-state features)
Each animal's mean P(AD) against its median recording age, shaded by sex, with a descriptive linear fit per genotype. Descriptive only — no sex adjustment, not evidence of progression.

In [ ]:
# =============================================================================
# XGBOOST FIGURE: Age vs model confidence (AD vs WT, combined features)
# (LEAVE-TWO-ANIMAL-OUT, caf42 excluded — uses cached l2o_results)
# =============================================================================
# Reuses the per-animal aggregation from the L2O training in cell 4.
# Each animal gets a single P(AD) (mean across its 9 L2O pairings).

from matplotlib.lines import Line2D

if 'l2o_results' not in globals():
    raise RuntimeError('Run the L2O training cell first — l2o_results required.')

# Aggregate the All/All-States L2O run to one record per animal
fold_res_all = l2o_results.get(('ALL', 'state', 'combined'))
if not fold_res_all:
    raise RuntimeError('Missing L2O results for ("ALL", "state", "combined").')

agg = _aggregate_by_animal_xgb(fold_res_all)

# Pull per-animal sex / age / true geno from df
age_meta = df[['animal', 'age']].drop_duplicates().groupby('animal')['age'].agg(['median', 'min', 'max'])
animal_meta = (df.drop_duplicates(subset=['animal'])
                 .set_index('animal')[['age', 'sex', 'genotype']]
                 .to_dict('index'))

animal_rows = []
for r in agg:
    a = r['animal']
    meta = animal_meta.get(a, {})
    sex_raw = str(meta.get('sex', '?')).lower()
    sex = 'female' if sex_raw.startswith('f') else 'male' if sex_raw.startswith('m') else 'unknown'
    pred_geno = 'AD' if r['mean_prob'] >= 0.5 else 'WT'
    animal_rows.append({
        'animal': a,
        'age': float(age_meta.loc[a, 'median']),
        'age_min': float(age_meta.loc[a, 'min']),
        'age_max': float(age_meta.loc[a, 'max']),
        'sex': sex,
        'geno': r['geno'],
        'mean_prob': r['mean_prob'],
        'pred': pred_geno,
        'correct': (pred_geno == r['geno']),
    })

# ── Plot ────────────────────────────────────────────────────────────────────
print('Mean uncalibrated AD score per animal (held-out, averaged across L2O pairs) against median '
      'recorded age; bars show the recorded age range and shade marks sex. Lines are descriptive '
      'least-squares fits per genotype (no sex adjustment). Black outline = misclassified.')
SEX_KEY = {'female': 'F', 'male': 'M'}
fig, ax = plt.subplots(figsize=(4.6, 3.4))
ax.axhline(0.5, color='black', ls='--', lw=0.8, zorder=1)
ax.axvline(AGE_CUTOFF_DAYS, color=REF_GREY, ls=':', lw=0.8, zorder=1)
ax.text(AGE_CUTOFF_DAYS, 1.02, f' {AGE_CUTOFF_DAYS} d', color=REF_GREY, fontsize=7, ha='left', va='top')

slope_text = {}
for geno in ['AD', 'WT']:
    pts = [r for r in animal_rows if r['geno'] == geno]
    if len(pts) >= 2:
        x = np.array([r['age'] for r in pts], dtype=float)
        yv = np.array([r['mean_prob'] for r in pts], dtype=float)
        p = np.polyfit(x, yv, 1)
        xline = np.linspace(x.min() - 10, x.max() + 10, 100)
        ax.plot(xline, p[0] * xline + p[1], color=GENO_COLORS[geno], lw=1.5, zorder=2)
        # Descriptive fitted line only; no inferential confidence band.
        slope_text[geno] = p[0] * 100

for r in animal_rows:
    color = SEX_COLORS.get((r['geno'], SEX_KEY.get(r['sex'])), '#888888')
    ax.errorbar(r['age'], r['mean_prob'],
                xerr=[[r['age'] - r['age_min']], [r['age_max'] - r['age']]],
                fmt='none', ecolor='#BDBDBD', elinewidth=0.8, zorder=2)
    ax.scatter(r['age'], r['mean_prob'], marker='^' if r['geno'] == 'AD' else 'o', s=32, color=color,
               edgecolors='none' if r['correct'] else 'black', linewidths=1.0, zorder=4)
    ax.annotate(r['animal'], xy=(r['age'], r['mean_prob']), xytext=(4, 0), textcoords='offset points',
                fontsize=6, color='#6E6E6E', va='center')

# Limits & labels based on the animals we still have (caf42 excluded)
plot_ages = np.array([r['age'] for r in animal_rows])
ax.set_xlim(plot_ages.min() - 20, plot_ages.max() + 40)
ax.set_ylim(0, 1.02)
ax.set_xlabel('Age (days; median of recordings, bars = range)')
ax.set_ylabel('Mean AD score (uncalibrated)')
ax.set_title('All animals, all-state features, by sex')

ad_slope = slope_text.get('AD', np.nan)
wt_slope = slope_text.get('WT', np.nan)
stat_box(ax, f'Slope per 100 d\nAD {ad_slope:+.3f}\nWT {wt_slope:+.3f}', x=1.03, y=0.0, ha='left', va='bottom')

legend_handles = [
    Line2D([], [], color=SEX_COLORS[(g, s)], marker='^' if g == 'AD' else 'o', ls='', ms=5,
           label=f'{g} {"female" if s == "F" else "male"}')
    for g, s in [('AD', 'F'), ('AD', 'M'), ('WT', 'F'), ('WT', 'M')]
] + [
    Line2D([], [], color='white', marker='o', markeredgecolor='black', ls='', ms=5, label='Misclassified'),
    Line2D([], [], color='black', ls='--', lw=0.8, label='Decision threshold'),
]
ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.savefig('fig_xgb_age_confidence.png')
plt.show()
print('Saved: fig_xgb_age_confidence.png')
